# Tools 工具
## Create tools  创建工具
### Basic tool definition  基本工具定义
使用 @tool 装饰器。默认情况下，函数的文档字符串会成为工具的描述，帮助模型理解何时使用该工具：

In [ ]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

类型提示是必需的 ，因为它们定义了工具的输入模式。文档字符串应简洁明了，以便模型理解工具的用途。

### Customize tool properties 自定义工具属性
#### Custom tool name  自定义工具名称

In [ ]:
# 默认情况下，工具名称来源于函数名称。如果需要更具描述性的名称，请对其进行覆盖
@tool("web_search")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)  # web_search

#### Custom tool description  自定义工具描述

In [ ]:
# 覆盖自动生成的工具描述，以便获得更清晰的模型指导
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

### Advanced schema definition 高级模式定义

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [ ]:
weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

@tool(args_schema=weather_schema)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

### Reserved argument names  保留参数名称
- config 保留用于将 RunnableConfig 传递给内部工具
- runtime 保留给 ToolRuntime 参数（访问状态、上下文、存储）

## Access context  访问上下文
工具在能够访问运行时信息（例如对话历史记录、用户数据和持久内存）时，其功能最为强大。

工具可以通过 ToolRuntime 参数访问运行时信息，该参数提供：
| Component | 描述 | 用例 |
|------------|------|------|
| **State（状态）** | 短期记忆 —— 当前对话中存在的可变数据（messages、counters、自定义字段等） | 访问对话历史记录，追踪工具调用次数 |
| **Context（语境）** | 调用时传递的不可变配置（user_id、session 信息等） | 根据用户身份个性化回复 |
| **Store（存储）** | 长期记忆 —— 可跨对话持久化的数据 | 保存用户偏好设置，维护知识库 |
| **Stream Writer（流式写入器）** | 在工具执行期间发出实时更新 | 展示长时间运行任务的进度 |
| **Config（配置）** | 执行过程中的 `RunnableConfig` | 访问 callbacks、tags、metadata |
| **Tool Call ID（工具调用 ID）** | 当前工具调用的唯一标识符 | 在日志和模型调用中关联对应的工具调用 |

### Short-term memory (State) 短期记忆（状态）
状态代表对话期间存在的短期记忆。它包括消息历史记录以及您在图状态中定义的任何自定义字段。
> 要访问工具状态，请在工具签名中添加 runtime: ToolRuntime 。此参数会自动注入并对 LLM 隐藏，不会出现在工具的架构中。
#### Access state  访问状态

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

> runtime 参数对模型是隐藏的。以上述示例为例，模型只能在工具模式中看到 pref_name 。

#### Update state  更新状态
使用 Command 更新代理的状态
> 当工具更新状态变量时，请考虑为这些字段定义一个 reducer 。由于 LLM 可以并行调用多个工具，reducer 可以确定如何解决并发工具调用更新同一状态字段时产生的冲突。

In [ ]:
from langgraph.types import Command
from langchain.tools import tool

@tool
def set_user_name(new_name: str) -> Command:
    """Set the user's name in the conversation state."""
    return Command(update={"user_name": new_name})

### Context  语境
上下文提供在调用时传递的不可变配置数据。可用于用户 ID、会话详细信息或在对话过程中不应更改的应用程序特定设置。

通过 runtime.context 访问上下文：

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("QWEN_BASE_URL")
api_key = os.getenv("QWEN_API_KEY")

model = ChatOpenAI(
    model="glm-4.7",
    base_url=base_url,
    api_key=api_key,
    temperature=0.7
)

In [14]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant."
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    context=UserContext(user_id="user123214")
)
result['messages'][-1].content

"I wasn't able to retrieve your account information. This could mean you haven't created an account yet, or there might be an authentication issue. \n\nTo check your current balance, you'll need to have an active account. If you already have an account, please make sure you're properly logged in. If you don't have an account yet, you'll need to create one first.\n\nWould you like help with creating an account, or do you need assistance with logging in to an existing account?"

### Long-term memory (Store)  长期记忆（存储）
`BaseStore` 提供持久存储，可在会话之间保留数据。与状态（短期记忆）不同，保存到存储中的数据在以后的会话中仍然可用。

通过 runtime.store 访问数据存储。该数据存储使用命名空间/键模式来组织数据：

In [18]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})['messages'][-1].content

# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

"Here's the user information for ID 'abc123':\n\n- **Name:** Foo\n- **Age:** 25\n- **Email:** foo@langchain.dev"

### Stream writer  流媒体撰稿人
在工具执行过程中，实时传输工具的更新信息。这对于在长时间运行的操作期间向用户提供进度反馈非常有用。

使用 runtime.stream_writer 发出自定义更新：

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    # Stream custom updates as the tool executes
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")

    return f"It's always sunny in {city}!"

## ToolNode  工具节点
ToolNode 是一个预构建节点，用于在 LangGraph 工作流中执行工具。它可以自动处理并行工具执行、错误处理和状态注入。
> 对于需要对工具执行模式进行精细控制的自定义工作流程，请使用 ToolNode 而不是 create_agent 。它是驱动代理工具执行的基础模块。

In [ ]:
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

# Create the ToolNode with your tools
tool_node = ToolNode([search, calculator])

# Use in a graph
builder = StateGraph(MessagesState)
builder.add_node("tools", tool_node)
# ... add other nodes and edges

### Tool return values  工具返回值
- 返回一个 string ，以便人类阅读结果。

流程:
> - 返回值被转换为 ToolMessage 。
> - 模型识别到该文本后，会决定下一步该做什么。
> - 除非模型或其他工具稍后进行更改，否则不会更改任何代理状态字段。

当结果为自然可读的文本时，请使用此选项。


In [ ]:
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is currently sunny in {city}."

- 返回一个 object ，其中包含模型需要解析的结构化结果。

当你的工具生成模型应该检查的结构化数据时，返回一个对象（例如， dict ）。

当下游推理受益于显式字段而不是自由格式文本时，请使用此功能。

> - 该对象被序列化并作为工具输出发送回去。
> - 该模型可以读取特定字段并对其进行推理。
> - 与字符串返回一样，这不会直接更新图状态。

In [ ]:
from langchain.tools import tool


@tool
def get_weather_data(city: str) -> dict:
    """Get structured weather data for a city."""
    return {
        "city": city,
        "temperature_c": 22,
        "conditions": "sunny",
    }

- 当您需要写入状态时，返回一个带有可选消息的 Command 。

当工具需要更新图（Graph）的状态时（例如设置用户偏好或应用状态），应返回一个 **Command**。

你可以返回一个 **Command**：

* 可以包含 `ToolMessage`
* 也可以不包含 `ToolMessage`

如果模型需要知道工具执行成功（例如确认偏好已修改），
那么在更新中应包含一个 `ToolMessage`，并且：

* 使用 `runtime.tool_call_id` 作为 `tool_call_id` 参数

这样模型才能正确关联这次工具调用的结果。


In [ ]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Set the preferred response language."""
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Language set to {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

- 该命令使用 update 更新状态。
- 更新后的状态可供同一运行中的后续步骤使用。
- 对于可能被并行工具调用更新的字段，请使用 reducer。

当工具不仅返回数据，而且还改变代理状态时，请使用此功能。

### Error handling  错误处理

In [ ]:
from langgraph.prebuilt import ToolNode

tools = []

# 默认：捕获调用错误，重新抛出执行错误
tool_node = ToolNode(tools)

# 捕获所有错误并将错误消息返回给 LLM
tool_node = ToolNode(tools, handle_tool_errors=True)

# 自定义错误消息
tool_node = ToolNode(tools, handle_tool_errors="出现错误，请重试。")

# 自定义错误处理器
def handle_error(e: ValueError) -> str:
    return f"无效输入：{e}"

tool_node = ToolNode(tools, handle_tool_errors=handle_error)

# 仅捕获特定异常类型
tool_node = ToolNode(tools, handle_tool_errors=(ValueError, TypeError))

### Route with tools_condition 使用 tools_condition 进行路由
使用 `tools_condition` 根据 LLM 是否发出工具调用进行条件路由：

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, MessagesState, START, END

call_llm = lambda state: "LLM response"

builder = StateGraph(MessagesState)
builder.add_node("llm", call_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)  # Routes to "tools" or END
builder.add_edge("tools", "llm")

graph = builder.compile()

### State injection  状态注入
工具可以通过 `ToolRuntime `访问当前图状态：

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.prebuilt import ToolNode

@tool
def get_message_count(runtime: ToolRuntime) -> str:
    """Get the number of messages in the conversation."""
    messages = runtime.state["messages"]
    return f"There are {len(messages)} messages."

tool_node = ToolNode([get_message_count])

## Prebuilt tools  预构建工具
LangChain 提供了一系列预构建的工具和工具包，用于执行诸如网页搜索、代码解析、数据库访问等常见任务。这些即用型工具可以直接集成到您的代理中，无需编写自定义代码。

## Server-side tool use  服务器端工具的使用
某些聊天模型内置了一些工具，这些工具由模型提供商在服务器端执行。这些工具包括网页搜索和代码解释器等功能，无需您定义或托管工具逻辑。